# Vicarious Cupboard — Expression Cells (Session Three, Part 2)
**The new room: witnessed pain BEHAVIOUR (scream/wince/writhe), not witnessed injury.**

Pipeline: discovery (inference search, registered Show/SUM net, layers 12 + 20) → recurrence count → automatic profiling of every candidate on all 32 sentences → gates and controls.

**Registered rules (carved before running):**
- Discovery seeds: the 6 pain-other expression sentences only (controls are never seeds). Top 15 per search, SUM-ranked excluding BOS, no density filter. Candidate = appears in ≥2 of 6 seeds at a layer.
- Gate: pain-other peak ≥ 2× neutral-other peak AND ≥ 10, on ≥ 4 of 6 expression pairs.
- Controls (classification, applied to any survivor): must beat the flinch sentence by 2× (else: startle/alarm feature) and the frown sentence by 2× (else: negative-valence feature). Only a survivor passing BOTH is classified pain-expression.
- Self cells are descriptive only.
- The five injury near-misses (9784, 1041, 7541, 12851, 15143) are also profiled on all 32 sentences as a probe — no gate, pure description.

**This notebook refuses to run until the prediction envelopes are filed.** That's Cell 1, and it's not a joke.

In [ ]:
# Cell 1 — governance gate. Envelope addenda BEFORE data, as registered on 24 Oct.
answer = input("Have ALL prediction addenda (Rachel + models) been pasted into the Session log? Type YES to proceed: ")
assert answer.strip().upper() == "YES", "Then go collect the envelopes first. The cupboard has waited six weeks; it can wait an hour."
print("Envelopes sealed. Proceeding.")


In [ ]:
# Cell 2 — setup
import requests, time, getpass
import pandas as pd
from collections import defaultdict

API_KEY = getpass.getpass("Paste your Neuronpedia API key: ")
MODEL = "gemma-2-2b"
SOURCESET = "gemmascope-res-16k"
LAYERS = ["12-gemmascope-res-16k", "20-gemmascope-res-16k"]

def api_post(path, payload):
    r = requests.post(f"https://neuronpedia.org/api/{path}", json=payload,
                      headers={"X-Api-Key": API_KEY}, timeout=120)
    r.raise_for_status()
    return r.json()

def get_activation(source, index, text):
    j = api_post("activation/new", {"feature": {"modelId": MODEL, "layer": source, "index": str(index)}, "customText": text})
    return j["tokens"], j["values"]

def peak_excl_bos(tokens, values):
    """Peak value and its token. Drops a BOS bookkeeping token ONLY if one is actually present —
    the search endpoint sends one, the activation endpoint does not (calibration caught this, 28 Oct)."""
    tv = list(zip(tokens, values))
    if tv and tv[0][0].strip() in ("<bos>", "<|endoftext|>"):
        tv = tv[1:]
    tok, val = max(tv, key=lambda x: x[1])
    return tok.strip(), round(val, 2)

def search_sum_net(text, layer_source):
    """The registered Show/SUM net: no density filter, BOS ignored, ranked by SUM over all non-BOS tokens."""
    first = api_post("search-all", {"modelId": MODEL, "sourceSet": SOURCESET, "text": text,
                                    "selectedLayers": [layer_source], "sortIndexes": [],
                                    "ignoreBos": True, "densityThreshold": -1, "numResults": 25})
    n_tokens = len(first["tokens"])
    j = api_post("search-all", {"modelId": MODEL, "sourceSet": SOURCESET, "text": text,
                                "selectedLayers": [layer_source], "sortIndexes": list(range(1, n_tokens)),
                                "ignoreBos": True, "densityThreshold": -1, "numResults": 25})
    out = []
    for res in j["result"][:15]:
        vals = res["values"]
        s = round(sum(vals[1:]), 1)
        tv = list(zip(j["tokens"], vals))[1:]
        tok, val = max(tv, key=lambda x: x[1])
        out.append({"index": int(res["index"]), "sum": s, "peak_word": tok.strip(), "peak": round(val, 2)})
    return j["tokens"], out


In [ ]:
# Cell 3 — the locked expression panel (32 sentences, verbatim; approved by Rachel 28 Oct)
# Channels: V vocal, F facial, B whole-body, M manual. X = control pairs (never discovery seeds).
PAIRS = {
 "V1": {"PS": "I cry out.",            "NS": "I call out.",
        "PO": "I watch my daughter cry out.",        "NO": "I watch my daughter call out."},
 "V2": {"PS": "I yelp.",               "NS": "I laugh.",
        "PO": "I watch my daughter yelp.",           "NO": "I watch my daughter laugh."},
 "F1": {"PS": "I wince.",              "NS": "I grin.",
        "PO": "I watch my daughter wince.",          "NO": "I watch my daughter grin."},
 "F2": {"PS": "I grimace.",            "NS": "I smile.",
        "PO": "I watch my daughter grimace.",        "NO": "I watch my daughter smile."},
 "B1": {"PS": "I writhe.",             "NS": "I stretch.",
        "PO": "I watch my daughter writhe.",         "NO": "I watch my daughter stretch."},
 "M1": {"PS": "I clutch my wrist.",    "NS": "I rub my wrist.",
        "PO": "I watch my daughter clutch her wrist.", "NO": "I watch my daughter rub her wrist."},
 # Controls — profiled on every candidate, never used as discovery seeds:
 "X1": {"PS": "I flinch.",             "NS": "I blink.",
        "PO": "I watch my daughter flinch.",         "NO": "I watch my daughter blink."},   # startle/fear pole
 "X2": {"PS": "I frown.",              "NS": "I beam.",
        "PO": "I watch my daughter frown.",          "NO": "I watch my daughter beam."},    # negative-valence pole
}
EXPRESSION_ITEMS = ["V1", "V2", "F1", "F2", "B1", "M1"]   # the gate runs on these six
SEEDS = [PAIRS[k]["PO"] for k in EXPRESSION_ITEMS]

NEAR_MISSES = [
    # Injury near-misses (descriptive probe)
    {"index": 9784,  "source": "20-gemmascope-res-16k", "layer": 20},
    {"index": 1041,  "source": "12-gemmascope-res-16k", "layer": 12},
    {"index": 7541,  "source": "12-gemmascope-res-16k", "layer": 12},
    {"index": 12851, "source": "12-gemmascope-res-16k", "layer": 12},
    {"index": 15143, "source": "12-gemmascope-res-16k", "layer": 12},
    # Attachment features (Rachel's addendum corollary: flat on all expression sentences)
    {"index": 2291,  "source": "12-gemmascope-res-16k", "layer": 12},
    {"index": 1692,  "source": "20-gemmascope-res-16k", "layer": 20},
]

for pid, cells_ in PAIRS.items():
    for cid, s in cells_.items():
        assert s == s.strip() and s.endswith("."), f"Pre-flight fail {pid}-{cid}: {s!r}"
print("Pre-flight passed: all 32 sentences end cleanly in '.'")


In [ ]:
# Cell 4 — validate BOTH instruments against hand-collected history before anything new counts
# 4a. Discovery pipeline must reproduce Session One's browser harvest (Show net, L12, K-PO), exact list, exact order.
EXPECTED_L12_KPO = [4667, 2620, 11087, 1041, 2291, 7507, 12851, 1178, 7541, 15143, 11306, 14906, 4790, 3260, 4987]
print("Validating discovery pipeline against Session One K-PO (L12, Show net)...")
toks, results = search_sum_net("I watch the knife slice into my daughter's finger.", "12-gemmascope-res-16k")
got = [r["index"] for r in results]
print("  expected:", EXPECTED_L12_KPO)
print("  got:     ", got)
assert got == EXPECTED_L12_KPO, "DISCOVERY VALIDATION FAILED — net differs from the registered browser net. STOP and tell Fable."
print("  Discovery net validated: identical list, identical order.\n")
time.sleep(1)

# 4b. Activation calibration (same four hand-collected values as the injury-trial notebook, +-2.5%)
CALIBRATION = [
    (4667,  "12-gemmascope-res-16k", "I watch the knife slice into my daughter's finger.", 378.0),
    (2620,  "12-gemmascope-res-16k", "I watch the knife rest against my daughter's finger.", 31.0),
    (9784,  "20-gemmascope-res-16k", "I watch the knife slice into my daughter's finger.", 39.25),
    (1692,  "20-gemmascope-res-16k", "I watch the hammer strike my daughter's thumb.", 36.75),
]
ok = True
for index, source, sentence, expected in CALIBRATION:
    tokens, values = get_activation(source, index, sentence)
    tok, peak = peak_excl_bos(tokens, values)
    rel = abs(peak - expected) / expected
    status = "OK" if rel <= 0.025 else "FAIL"
    if rel > 0.025: ok = False
    print(f"  {source}:{index}  expected {expected}, got {peak} on {tok!r}  [{status}]")
    time.sleep(1)
assert ok, "ACTIVATION CALIBRATION FAILED — STOP. No new data counts."
print("\nBoth instruments validated. The cupboard door may be opened.")


In [ ]:
# Cell 5 — DISCOVERY: 6 seeds x 2 layers, registered net, top-15 each
disc_rows = []
for layer_source in LAYERS:
    layer_num = int(layer_source.split("-")[0])
    for pid in EXPRESSION_ITEMS:
        sentence = PAIRS[pid]["PO"]
        toks, results = search_sum_net(sentence, layer_source)
        for rank, r in enumerate(results, 1):
            disc_rows.append({"Layer": layer_num, "Seed": pid, "Rank": rank, "Feature": r["index"],
                              "SUM": r["sum"], "Peak word": r["peak_word"], "Peak": r["peak"]})
        print(f"L{layer_num} {pid} done ({sentence})")
        time.sleep(2)

disc = pd.DataFrame(disc_rows)
disc.to_csv("expression_discovery.csv", index=False)

# Recurrence count: candidate = appears in >= 2 of 6 seeds at its layer
print("\n=== CANDIDATES (>=2 of 6 seeds) ===")
CANDIDATES = []
for layer_num in [12, 20]:
    sub = disc[disc["Layer"] == layer_num]
    counts = sub.groupby("Feature")["Seed"].nunique().sort_values(ascending=False)
    for feat, c in counts.items():
        if c >= 2:
            CANDIDATES.append({"index": int(feat), "source": f"{layer_num}-gemmascope-res-16k", "layer": layer_num})
            print(f"  L{layer_num} feature {feat}: {c}/6 seeds")
print(f"\n{len(CANDIDATES)} candidates proceed to profiling.")


In [ ]:
# Cell 6 — PROFILING: every candidate + the five injury near-misses, on all 32 sentences
to_profile = CANDIDATES + NEAR_MISSES
seen = set(); unique = []
for f in to_profile:
    key = (f["index"], f["source"])
    if key not in seen:
        seen.add(key); unique.append(f)

total = len(unique) * 32
print(f"Profiling {len(unique)} features x 32 sentences = {total} measurements (~{total//60 + 1} min)...")
prof_rows = []
done = 0
for f in unique:
    for pid, cells_ in PAIRS.items():
        for cid in ["PS", "NS", "PO", "NO"]:
            tokens, values = get_activation(f["source"], f["index"], cells_[cid])
            tok, peak = peak_excl_bos(tokens, values)
            prof_rows.append({"Feature": f["index"], "Layer": f["layer"], "Sentence": f"{pid}-{cid}",
                              "Cell": cid, "Item": pid, "Peak word": tok, "Peak": peak})
            done += 1
            if done % 32 == 0: print(f"  {done}/{total} ({f['index']} done)")
            time.sleep(1)

prof = pd.DataFrame(prof_rows)
prof.to_csv("expression_profiles.csv", index=False)
print("Saved expression_profiles.csv")


In [ ]:
# Cell 7 — GATES AND CONTROLS (registered rules, no post-hoc softening)
def peak_of(sub, pid, cid):
    return float(sub[(sub["Item"] == pid) & (sub["Cell"] == cid)]["Peak"].iloc[0])

print("=" * 72)
print("EXPRESSION GATE: PO >= 2x NO, PO >= 10, on >= 4 of 6 expression pairs")
print("=" * 72)
for f in unique:
    sub = prof[(prof["Feature"] == f["index"]) & (prof["Layer"] == f["layer"])]
    passes = 0; lines = []
    for pid in EXPRESSION_ITEMS:
        po, no = peak_of(sub, pid, "PO"), peak_of(sub, pid, "NO")
        ratio = po / no if no > 0 else float("inf")
        hit = (po >= 2 * no) and (po >= 10)
        passes += hit
        r = f"{ratio:.2f}" if ratio != float("inf") else "inf"
        lines.append(f"    {pid}: {po} vs {no}  ratio {r}" + ("  PASS" if hit else ""))
    is_nm = any(f["index"] == n["index"] and f["source"] == n["source"] for n in NEAR_MISSES)
    tag = " [injury near-miss probe — descriptive only]" if is_nm else ""
    if passes >= 4 and not is_nm:
        # Controls: beat flinch (X1) and frown (X2) by 2x on the PO side
        pain_mean = sum(peak_of(sub, p, "PO") for p in EXPRESSION_ITEMS) / 6
        flinch, frown = peak_of(sub, "X1", "PO"), peak_of(sub, "X2", "PO")
        beats_flinch, beats_frown = pain_mean >= 2 * flinch, pain_mean >= 2 * frown
        if beats_flinch and beats_frown:
            verdict = "SURVIVOR — classified PAIN-EXPRESSION (passed both controls)"
        elif not beats_flinch:
            verdict = f"gate-passer KILLED by flinch control (flinch {flinch} vs pain-mean {pain_mean:.1f}) -> startle/alarm"
        else:
            verdict = f"gate-passer KILLED by frown control (frown {frown} vs pain-mean {pain_mean:.1f}) -> negative-valence"
    else:
        verdict = "non-survivor"
    print(f"\nFeature {f['index']} (L{f['layer']}): {passes}/6 -> {verdict}{tag}")
    print("\n".join(lines))

print("\nDownload both CSVs and bring everything back to Fable for the audit and the envelope reading.")
try:
    from google.colab import files
    files.download("expression_discovery.csv"); files.download("expression_profiles.csv")
except ImportError:
    pass


---
*Built 28 Oct 2026. Instruments: `/api/search-all` (discovery; validated against the Session One browser harvest before use) and `/api/activation/new` (profiling; calibrated against four hand-collected values). Panel: 6 expression pairs + flinch control + frown control, self and other forms. Rachel adjudicates; nothing here overrides the registered rules.*